In [24]:
import pandas as pd

covariates    = pd.read_csv("Data/covariates_deployment_dataset_2026-03-17.csv")
corrected_ids = pd.read_csv("Input_phase_2/Correction_crm_users.csv")["user_id"].drop_duplicates()

def load_model(treatment_path, control_path, model_name):
    treatment = pd.read_csv(treatment_path)
    control   = pd.read_csv(control_path)
    control["original_incentive_name"] = "control_" + control["original_incentive_name"].astype(str)
    df = pd.concat([treatment, control], ignore_index=True).merge(covariates, on="customer_nk", how="inner")
    df["in_correction"] = df["customer_nk"].isin(corrected_ids).astype(int)
    df["model"] = model_name
    return df

df = pd.concat([
    load_model("Input_phase_2/treatment_selected_binary_2.csv", "Input_phase_2/control_selected_binary_2.csv", "binary"),
    load_model("Input_phase_2/treatment_selected_multi_2.csv",  "Input_phase_2/control_selected_multi_2.csv",  "multi"),
], ignore_index=True)

C:\Users\tsterk\AppData\Local\Temp\ipykernel_27536\4104012459.py:3: DtypeWarning: Columns (0: total_volume, 1: food_total, 2: sports_total, 3: monetary_value_52wk, 4: online_sales_52w, 5: retail_sales_52w, 6: monetary_value_53w_104w, 7: online_sales_53w_104w, 8: retail_sales_53w_104w) have mixed types. Specify dtype option on import or set low_memory=False.
  covariates    = pd.read_csv("Data/covariates_deployment_dataset_2026-03-17.csv")


In [27]:
def incentive_distribution_comparison(df: pd.DataFrame) -> pd.DataFrame:
    grp = ["model", "original_incentive_name"]
    result = (
        df.groupby(grp).size().rename("before")
        .to_frame()
        .join(df[df["in_correction"] == 1].groupby(grp).size().rename("after"))
        .fillna(0)
        .astype(int)
        .reset_index()
    )
    result["pct_decrease"] = ((result["before"] - result["after"]) / result["before"] * 100).round(1)
    return result.sort_values(["model", "before"], ascending=[True, False])
    
df_incentive_dist = incentive_distribution_comparison(df_combined)
df_incentive_dist

,model,original_incentive_name,before,after,pct_decrease
0,binary,BNLX_ChurnP_10_test_export.csv,2970,2349,20.9
1,binary,BNLX_ChurnP_10eu_test_export.csv,2970,2379,19.9
2,binary,BNLX_ChurnP_250_test_export.csv,2970,2401,19.2
3,binary,BNLX_ChurnP_25_test_export.csv,2970,2406,19.0
4,binary,BNLX_ChurnP_500_test_export.csv,2970,2445,17.7
5,binary,BNLX_ChurnP_5eu_test_export.csv,2970,2496,16.0
6,binary,BNLX_ChurnP_SKUe_test_export.csv,2970,2372,20.1
7,binary,control_BNLX_ChurnP_10_test_export.csv,2970,2593,12.7
8,binary,control_BNLX_ChurnP_10eu_test_export.csv,2970,2594,12.7
9,binary,control_BNLX_ChurnP_250_test_export.csv,2970,2637,11.2


In [16]:
def coerce_metrics_to_numeric(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    df = df.copy()

    df[cols] = (
        df[cols]
        .replace({",": ""}, regex=True)
        .apply(pd.to_numeric, errors="coerce")
    )
    return df

In [28]:
def compute_test_vs_control(df: pd.DataFrame) -> pd.DataFrame:
    df = coerce_metrics_to_numeric(df, ["frequency", "monetary_value", "recency"])
    metrics = ["frequency", "monetary_value", "recency"]
    output = []

    def make_row(model, metric, arm, tb, ta, cb, ca):
        _, p_val = stats.ttest_ind(ta_vals, ca_vals, equal_var=False) if len(ta_vals) > 1 and len(ca_vals) > 1 else (None, None)
        return {
            "model": model, "metric": metric, "arm": arm,
            "test_before": tb, "test_after": ta,
            "control_before": cb, "control_after": ca,
            "diff_before":     round(tb - cb, 1),
            "diff_after":      round(ta - ca, 1),
            "diff_pct_before": round((tb - cb) / cb * 100, 1) if cb != 0 else None,
            "diff_pct_after":  round((ta - ca) / ca * 100, 1) if ca != 0 else None,
            "p_value":         round(p_val, 3) if p_val is not None else None,
            "significance":    sig_stars(p_val),
        }

    for model_name, mdf in df.groupby("model"):
        is_control = mdf["original_incentive_name"].str.startswith("control_")
        arm_names  = mdf.loc[~is_control, "original_incentive_name"].unique()

        for metric in metrics:
            for arm in arm_names:
                ta_vals = mdf.loc[(mdf["original_incentive_name"] == arm)              & (mdf["in_correction"] == 1), metric].dropna()
                ca_vals = mdf.loc[(mdf["original_incentive_name"] == "control_" + arm) & (mdf["in_correction"] == 1), metric].dropna()
                tb_vals = mdf.loc[mdf["original_incentive_name"] == arm,                metric].dropna()
                cb_vals = mdf.loc[mdf["original_incentive_name"] == "control_" + arm,  metric].dropna()
                short   = arm.replace("BNLX_ChurnP_", "").replace("_test_export.csv", "")
                output.append(make_row(model_name, metric, short, round(tb_vals.mean(), 1), round(ta_vals.mean(), 1), round(cb_vals.mean(), 1), round(ca_vals.mean(), 1)))

            # TOTAL
            ta_vals = mdf.loc[~is_control & (mdf["in_correction"] == 1), metric].dropna()
            ca_vals = mdf.loc[ is_control & (mdf["in_correction"] == 1), metric].dropna()
            tb_vals = mdf.loc[~is_control, metric].dropna()
            cb_vals = mdf.loc[ is_control, metric].dropna()
            output.append(make_row(model_name, metric, "TOTAL", round(tb_vals.mean(), 1), round(ta_vals.mean(), 1), round(cb_vals.mean(), 1), round(ca_vals.mean(), 1)))

    return pd.DataFrame(output).sort_values(["model", "metric", "arm"]).reset_index(drop=True)

pd.set_option("display.float_format", "{:.1f}".format)
df_test_vs_control = compute_test_vs_control(df)
df_test_vs_control["p_value"] = df_test_vs_control["p_value"].map("{:.3f}".format)
df_test_vs_control

,model,metric,arm,test_before,test_after,control_before,control_after,diff_before,diff_after,diff_pct_before,diff_pct_after,p_value,significance
0,binary,frequency,10,5.6,5.5,5.5,5.4,0.1,0.1,1.8,1.9,0.495,
1,binary,frequency,10eu,4.8,4.7,4.8,4.7,0.0,0.0,0.0,0.0,0.832,
2,binary,frequency,25,3.8,3.6,4.0,3.8,-0.2,-0.2,-5.0,-5.3,0.299,
3,binary,frequency,250,4.4,4.4,4.6,4.4,-0.2,0.0,-4.3,0.0,0.897,
4,binary,frequency,500,4.3,4.1,4.2,3.9,0.1,0.2,2.4,5.1,0.481,
5,binary,frequency,5eu,3.4,3.3,3.4,3.3,0.0,0.0,0.0,0.0,0.798,
6,binary,frequency,SKUe,5.8,5.6,5.6,5.4,0.2,0.2,3.6,3.7,0.290,
7,binary,frequency,TOTAL,4.6,4.4,4.6,4.4,0.0,0.0,0.0,0.0,0.632,
8,binary,monetary_value,10,147.7,147.3,142.8,141.3,4.9,6.0,3.4,4.2,0.234,
9,binary,monetary_value,10eu,114.9,114.0,114.0,113.8,0.9,0.2,0.8,0.2,0.949,
